In [1]:
import cv2
import numpy as np
import os
import mediapipe as mp
import joblib
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
def name_map(x):
    if x == 'Real Image':
        return 0
    elif x == 'AI Generated Image':
        return 1

In [3]:
def get_name(x):
    if x == 0:
        return 'Real Image'
    elif x == 1:
        return 'AI Generated Image'

In [4]:
def detect_face(img):
    mp_face = mp.solutions.face_detection
    faces = mp_face.FaceDetection()

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces = faces.process(img)

    if not faces.detections:
        return None, None

    detection = faces.detections[0]
    bbox = detection.location_data.relative_bounding_box
    h_img, w_img, _ = img.shape

    x = int(bbox.xmin * w_img)
    y = int(bbox.ymin * h_img)
    w = int(bbox.width * w_img)
    h = int(bbox.height * h_img)

    x, y = max(0, x), max(0, y)
    w = min(w, w_img - x)
    h = min(h, h_img - y)

    face = img[y:y+h, x:x+w]
    face = cv2.resize(face, (128, 128))
    face = face / 255.0
    face = np.reshape(face, (1, 128, 128, 3))
    return face, (x, y, w, h)

In [5]:
def prepare_training_data(real_path, fake_path):
    data = []
    labels = []

    for img_name in os.listdir(real_path)[:3000]:
        image = cv2.imread(os.path.join(real_path, img_name))
        image = cv2.resize(image, (128, 128,))
        if image.shape != (128, 128, 3):
            continue
        data.append(image)
        labels.append(name_map('Real Image'))

    for img_name in os.listdir(fake_path)[:3000]:
        image = cv2.imread(os.path.join(fake_path, img_name))
        image = cv2.resize(image, (128, 128))
        if image.shape != (128, 128, 3):
            continue
        data.append(image)
        labels.append(name_map('AI Generated Image'))

    return data, labels

In [6]:
data, labels = prepare_training_data(
    r'C:\Users\hafee\OneDrive\Desktop\Desktop\AI_Powered Face Image Authentication System\Dataset\Train\Real',
    r'C:\Users\hafee\OneDrive\Desktop\Desktop\AI_Powered Face Image Authentication System\Dataset\Train\Fake'
)
print('Data Prepared...')
print(len(labels))

Data Prepared...
6000


In [7]:
data = np.array(data)
labels = np.array(labels)
data, labels = shuffle(data, labels, random_state=42)
data = data / 255.0
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
print(x_train.shape)
print(x_test.shape)

(4800, 128, 128, 3)
(1200, 128, 128, 3)


In [8]:
EarlyStopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

#convolution layer
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))
model.add(MaxPool2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPool2D((2, 2)))
model.add(Flatten())

#hidden Layer
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

c:\Users\hafee\OneDrive\Desktop\Desktop\AI_Powered Face Image Authentication System\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05
)
train_gen = datagen.flow(x_train, y_train)

model.fit(
    train_gen,
    epochs=50,
    validation_data=(x_test, y_test),
    callbacks=[EarlyStopping]
)
print('Working')

Epoch 1/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 37s 236ms/step - accuracy: 0.7825 - loss: 0.4629 - val_accuracy: 0.9108 - val_loss: 0.2015
Epoch 2/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 223ms/step - accuracy: 0.8910 - loss: 0.2635 - val_accuracy: 0.9367 - val_loss: 0.1608
Epoch 3/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 226ms/step - accuracy: 0.9092 - loss: 0.2113 - val_accuracy: 0.9450 - val_loss: 0.1399
Epoch 4/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 35s 231ms/step - accuracy: 0.9185 - loss: 0.2048 - val_accuracy: 0.9442 - val_loss: 0.1425
Epoch 5/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 224ms/step - accuracy: 0.9285 - loss: 0.1713 - val_accuracy: 0.9475 - val_loss: 0.1295
Epoch 6/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 224ms/step - accuracy: 0.9371 - loss: 0.1570 - val_accuracy: 0.9617 - val_loss: 0.1019
Epoch 7/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 228ms/step - accuracy: 0.9473 - loss: 0.1379 - val_accuracy: 0.9425 - val_loss: 0.1578
Epoch 8/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 33s 223ms/step - accuracy: 0.9438 - loss: 0

In [10]:
print("\n--- Final Results ---")
train_loss, train_acc = model.evaluate(x_train, y_train)
test_loss,  test_acc  = model.evaluate(x_test,  y_test)
print(f"Train Accuracy : {train_acc * 100:.2f}%")
print(f"Test Accuracy  : {test_acc  * 100:.2f}%")
print(f"Train Loss     : {train_loss:.4f}")
print(f"Test Loss      : {test_loss:.4f}")


--- Final Results ---
150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9946 - loss: 0.0146
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9850 - loss: 0.0443
Train Accuracy : 99.46%
Test Accuracy  : 98.50%
Train Loss     : 0.0146
Test Loss      : 0.0443


In [12]:
joblib.dump(model, "model.jbl")

['model.jbl']